In [1]:
!apt-get update
!apt-get install -y poppler-utils tesseract-ocr

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,803 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,309 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,842 kB]
Get:14 h

In [2]:
!pip install pytesseract pdfplumber pdf2image langchain-groq langgraph python-dotenv pandas

...Import all dependencies

In [2]:
import os
import json
import time
import glob
import asyncio
import sqlite3
from typing import TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from pdf2image import convert_from_path
import pdfplumber
import pytesseract
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
import pandas as pd

...Load Environment variables and Path

In [3]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("API Key loaded")

API Key loaded


...Define Pydantic Schema & Parser

In [4]:
class ClassificationOutput(BaseModel):
    classification: str = Field(description="Cease, Uncertain, or Irrelevant")
    confidence: float = Field(description="Confidence between 0 and 1")
    reason: str = Field(description="Short explanation")

parser = PydanticOutputParser(pydantic_object=ClassificationOutput)

...Define AgentState

In [5]:
class AgentState(TypedDict):
    file: str
    text: str
    classification: str
    confidence: float
    reason: str
    action: str
    current_file: str

...Initialize LLM

In [6]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

...Document Classifier

In [16]:
import re
def classify_document(state):
    text = state.get("text", "")

    if not text.strip():
        state.update({"classification": "Irrelevant", "confidence": 0.0, "reason": "Empty document"})
        return state

    prompt = f"""
    You are an expert legal document classifier specializing
    in Cease and Desist letters with 20 years of experience.

    TASK:
    Read the document carefully and classify it into one of three categories.
    Extract key details only if it is a valid Cease and Desist letter.

    CLASSIFICATION

    cease
    The document has all of the following:
    - Clear legal demands asking someone to stop an activity
    - Identifies both sender and receiver
    - References a specific violation or harm caused
    - States or implies legal consequences for non-compliance
    - Written by or on behalf of an attorney or rights holder

    uncertain
    The document has some legal language but:
    - Missing one or more key elements of a cease and desist
    - Demands are vague or not clearly asking to stop something
    - Sender or receiver cannot be clearly identified
    - Wording is ambiguous
    - Not confident enough to classify as cease or irrelevant

    irrelevant
    The document is not a cease and desist letter.
    Examples: invoice, authorization letter, business correspondence,
    contract, newsletter, complaint, or inquiry.

    CONFIDENCE SCORE
    0.80 to 1.00 → All key elements clearly present (very confident)
    0.50 to 0.79 → Most elements present but some missing.
    0.00 to 0.49 → Weak or vague → classify as uncertain or irrelevant

    IMPORTANT RULES
    - confidence must be a number between 0 and 1
    - confidence must be a decimal number like 0.72 (not a string)
    - NEVER assign 0.0 unless all elements are missing-
    - NEVER assign 1.0 unless all elements are explicitly present
    - If unsure → always choose "uncertain"
    - Do NOT over-classify as "cease"

    EXTRACTION

    Only extract fields when classification is "cease".
    Set extracted_data = null for "uncertain" and "irrelevant".

    STRICT RULES:
  - Do NOT include any explanation before or after JSON
  - Do NOT use markdown (no ```json)
  - Output must start with {{ and end with }}

    DOCUMENT:
    {text[:1500]}
    """

    result = llm.invoke(prompt)

    try:
      raw_output = result.content.strip()

      # Remove markdown ```json ``` if present
      cleaned = re.sub(r"```json|```", "", raw_output).strip()

      # Extract JSON block only
      json_match = re.search(r"\{.*\}", cleaned, re.DOTALL)
      if json_match:
          cleaned = json_match.group()

      parsed = json.loads(cleaned)

      state["classification"] = parsed.get("classification", "Uncertain").capitalize()
      conf = parsed.get("confidence", 0)
      state["confidence"] = float(conf)

    except Exception as e:
      print("\nRAW LLM OUTPUT:")
      print(result.content)   # DEBUG (important)

      state["classification"] = "Uncertain"
      state["confidence"] = 0.5
      state["reason"] = parsed.get("reason", "")

    return state

...Routing functions

In [17]:
def route_decision(state):
    CONFIDENCE_THRESHOLD = 0.8
    cls = state.get("classification", "").lower()
    conf = state.get("confidence", 0.0)

    if cls == "cease":
        return "auto" if conf >= CONFIDENCE_THRESHOLD else "review"
    elif cls == "irrelevant":
        return "archive"
    else:
        return "review"

def route_archive(state):
    return "archive" if "irrelevant" in state.get("classification","").lower() else "skip"

def post_review_router(state):
    cls = state.get("classification", "").lower()
    if state.get("action") == "Approved by Human":
        return "auto"
    elif cls == "irrelevant":
        return "archive"
    else:
        return "uncertain"

...Agents

In [18]:
results_store = []  # In-memory store
audit_store = []    # For runtime audit
AUDIT_LOG_FILE = "cease_docs_audit.log"

def human_review(state):
    hitl_file = state["current_file"]
    print(f"\nFile: {hitl_file}\nSuggested Classification: {state['classification']}\nConfidence: {state.get('confidence')}\nReason: {state.get('reason')}")

    user_input = input("Approve? (y = approve / n = correct): ").strip().lower()
    if user_input == "y":
        state["action"] = "Approved by Human"
    else:
        corrected = input("Enter correct classification (Cease/Irrelevant): ").strip()
        state.update({"classification": corrected, "action": "Corrected by Human"})
    return state

def db_agent(state):
    conn = sqlite3.connect("cease_documents.db", check_same_thread=False)
    cursor = conn.cursor()
    cursor.execute("""INSERT INTO cease_docs (file_name, classification, confidence, action, reason, timestamp)
                      VALUES (?, ?, ?, ?, ?, ?)""",
                   (state["current_file"], state.get("classification"), state.get("confidence",0),
                    state.get("action","Stored to DB"), state.get("reason",""), time.strftime("%Y-%m-%d %H:%M:%S")))
    conn.commit()
    conn.close()
    results_store.append({"File Name": state["current_file"], "Classification": state.get("classification"),
                          "Confidence": round(state.get("confidence",0),2), "Action": state.get("action"), "Reason": state.get("reason")})

    print(f"{state["current_file"]} → {state.get("classification")} | Conf: {round(state.get('confidence',0),2)} | Stored in DB")
    return state

def archive_agent(state):
    state["action"] = "Archived"
    db_agent(state)  # store archive in DB
    return state

def audit_logger(state):
    record = {
        "file_name": state["current_file"],
        "classification": state.get("classification", "Unknown"),
        "confidence": round(state.get("confidence",0.0),2),
        "action": state.get("action","Pending"),
        "reason": state.get("reason",""),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    audit_store.append(record)
    with open(AUDIT_LOG_FILE,"a",encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return state

...Workflow Graph

In [20]:
workflow = StateGraph(AgentState)
workflow.add_node("classify", classify_document)
workflow.add_node("decision", lambda state: state)
workflow.add_node("db_agent", db_agent)
workflow.add_node("archive_agent", archive_agent)
workflow.add_node("human_review", human_review)
workflow.add_node("audit", audit_logger)

workflow.set_entry_point("classify")
workflow.add_edge("classify", "decision")

workflow.add_conditional_edges("decision", route_decision, {"auto": "db_agent", "review": "human_review", "archive": "archive_agent"})
workflow.add_conditional_edges("db_agent", route_archive, {"archive":"archive_agent", "skip":"audit"})
workflow.add_conditional_edges("human_review", post_review_router, {"auto":"db_agent", "archive":"archive_agent","uncertain":END})
workflow.add_edge("archive_agent","audit")
workflow.add_edge("audit",END)

app = workflow.compile()

...PDF Text Extraction & Image Preprocessing

In [21]:
def preprocess_image(pil_image):
    import cv2
    import numpy as np
    img = np.array(pil_image)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray,3)
    thresh = cv2.adaptiveThreshold(gray,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,11,2)
    return thresh

def extract_text_from_pdf(file_path):
    text = ""
    try:
        with pdfplumber.open(file_path) as pdf:
            for i,page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text and page_text.strip():
                    text += page_text + "\n"
                else:
                    images = convert_from_path(file_path, first_page=i+1, last_page=i+1)
                    for img in images:
                        text += pytesseract.image_to_string(preprocess_image(img)) + "\n"
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
    return text.strip()

...Document Loader & Batch Processing

In [22]:
def load_documents(folder):
    docs = []
    for file in glob.glob(os.path.join(folder,"*.pdf")):
        text = extract_text_from_pdf(file)
        if not text:
            print(f"No text extracted: {file}")
        docs.append({"file": os.path.basename(file), "text": text})
    return docs

async def process_document(doc):
    state = {"file": doc["file"], "text": doc["text"], "current_file": doc["file"]}
    result = app.invoke(state)
    return result

async def process_batch(folder):
    docs = load_documents(folder)
    for doc in docs:
        await process_document(doc)

...Database Initialization & Display

In [23]:
def init_db():
    conn = sqlite3.connect("cease_documents.db", check_same_thread=False)
    cursor = conn.cursor()
    cursor.execute("""CREATE TABLE IF NOT EXISTS cease_docs (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        file_name TEXT,
                        classification TEXT,
                        confidence REAL,
                        action TEXT,
                        reason TEXT,
                        timestamp TEXT)""")
    cursor.execute("DELETE FROM cease_docs")
    conn.commit()
    conn.close()

def show_table_from_db():
    conn = sqlite3.connect("cease_documents.db", check_same_thread=False)
    df = pd.read_sql_query("SELECT id as ID, file_name as 'File Name', classification as 'Classification', ROUND(confidence,2) as 'Confidence', action as 'Action', reason as 'Reason' FROM cease_docs", conn)
    conn.close()
    print("\n=== FINAL DB OUTPUT ===\n")
    try:
        print(df.to_markdown(index=False))
    except:
        print(df.to_string(index=False))

...Run Pipeline

In [24]:
init_db()
await process_batch("/content/InputFolder")
show_table_from_db()

LOA2.pdf → Irrelevant | Conf: 0.0 | Stored in DB

File: notice_1.pdf
Suggested Classification: Uncertain
Confidence: 0.62
Reason: None
Approve? (y = approve / n = correct): y
notice_1.pdf → Uncertain | Conf: 0.62 | Stored in DB

File: bw_doc_1.pdf
Suggested Classification: Uncertain
Confidence: 0.42
Reason: None
Approve? (y = approve / n = correct): n
Enter correct classification (Cease/Irrelevant): Irrelevant
bw_doc_1.pdf → Irrelevant | Conf: 0.42 | Stored in DB

File: bw_doc_3.pdf
Suggested Classification: Uncertain
Confidence: 0.42
Reason: None
Approve? (y = approve / n = correct): n
Enter correct classification (Cease/Irrelevant): Irrelevant
bw_doc_3.pdf → Irrelevant | Conf: 0.42 | Stored in DB

File: notice_3.pdf
Suggested Classification: Uncertain
Confidence: 0.0
Reason: None
Approve? (y = approve / n = correct): n
Enter correct classification (Cease/Irrelevant): irrelevant
notice_3.pdf → irrelevant | Conf: 0.0 | Stored in DB
LOA7.pdf → Cease | Conf: 0.95 | Stored in DB

File: bw